In [67]:
# exp01_all_noise_agent1_vs_baselines_mae_vs_nl_ijcai.py
# IJCAI two-column template, single-column figure: \includegraphics[width=\linewidth]{...}

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator, FuncFormatter, FormatStrFormatter


PATHS = {
    "iq_chain": "exp01_ibm_baseline+agent__iq_chain__mean_abs_err_MHz(1).csv",
    "chain": "exp01_ibm_baseline+agent__chain__mean_abs_err_MHz(1).csv",
    "realistic_chain": "exp01_ibm_baseline+agent__realistic_chain__mean_abs_err_MHz(1).csv",
}

OUT_PNG = "exp01_all_noise_agent1_vs_baselines_mae_vs_nl_ijcai.png"
OUT_PDF = "exp01_all_noise_agent1_vs_baselines_mae_vs_nl_ijcai.pdf"

# ---------------- Figure size (single-column) ----------------
FIG_W = 3.45
FIG_H = 2.25  # 2.15~2.40 can be tuned if needed

# ---------------- Tight margins (reduce left/right whitespace) ----------------
LEFT = 0.035
RIGHT = 0.995
BOTTOM = 0.22
TOP = 0.985

# Subplot spacing (smaller => each panel wider)
WSPACE = 0.14  # 0.10~0.18

# Legend row height ratio
LEGEND_RATIO = 0.24  # 0.20~0.30

# ---------------- Font sizes ----------------
FS_TITLE = 9
FS_LABEL = 9
FS_TICK = 8
FS_LEGEND = 8.2


def load_table(csv_path: str):
    df = pd.read_csv(csv_path)
    id_col = df.columns[0]
    x_cols = [c for c in df.columns if c != id_col]

    xs = np.array(sorted([float(c) for c in x_cols]))
    col_map = {float(c): c for c in x_cols}
    x_cols_ordered = [col_map[x] for x in xs]

    labels = df[id_col].astype(str).tolist()
    series = {
        lab: df.loc[i, x_cols_ordered].astype(float).to_numpy()
        for i, lab in enumerate(labels)
    }
    return xs, labels, series


def pick_agent_label(labels):
    for cand in labels:
        if cand.lower() in {"agent1", "a1", "onlyagent", "agent"}:
            return cand
    return "Agent1" if "Agent1" in labels else None


def main():
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "dejavuserif",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.minor.width": 0.6,
        "ytick.minor.width": 0.6,
    })

    tables = {k: load_table(v) for k, v in PATHS.items()}

    # global ranges
    global_ymax = 0.0
    x_min, x_max = None, None
    for _, (xs, _, series) in tables.items():
        global_ymax = max(global_ymax, max(float(np.nanmax(y)) for y in series.values()))
        x_min = float(np.min(xs)) if x_min is None else min(x_min, float(np.min(xs)))
        x_max = float(np.max(xs)) if x_max is None else max(x_max, float(np.max(xs)))

    # formatter: hide the rightmost tick label (0.10) for left/middle panels
    tol = max(1e-12, (x_max - x_min) * 1e-6)

    def fmt_hide_rightmost(x, pos):
        if abs(x - x_max) < tol:
            return ""
        return f"{x:.2f}"

    # collect all labels (consistent markers)
    all_labels = []
    for _, (_, labels, _) in tables.items():
        for l in labels:
            if l not in all_labels:
                all_labels.append(l)

    marker_cycle = ["o", "s", "^", "v", "P", "X", "D", "<", ">"]
    mk = {lab: marker_cycle[i % len(marker_cycle)] for i, lab in enumerate(all_labels)}
    mk["Agent1"] = "D"

    # --------- Figure + GridSpec: legend gets its OWN row (no overlap) ----------
    fig = plt.figure(figsize=(FIG_W, FIG_H))
    gs = fig.add_gridspec(
        2, 3,
        height_ratios=[LEGEND_RATIO, 1.0 - LEGEND_RATIO],
        left=LEFT, right=RIGHT, bottom=BOTTOM, top=TOP,
        wspace=WSPACE
    )

    ax_leg = fig.add_subplot(gs[0, :])
    ax_leg.axis("off")

    axes = [fig.add_subplot(gs[1, 0])]
    axes += [fig.add_subplot(gs[1, 1], sharey=axes[0])]
    axes += [fig.add_subplot(gs[1, 2], sharey=axes[0])]

    panel_order = ["iq_chain", "chain", "realistic_chain"]
    panel_titles = {"iq_chain": "iq chain", "chain": "chain", "realistic_chain": "realistic chain"}
    panel_letters = ["(a)", "(b)", "(c)"]

    handles_for_legend = {}

    for ax, key, letter in zip(axes, panel_order, panel_letters):
        xs, labels, series = tables[key]
        agent_label = pick_agent_label(labels)
        baseline_labels = [l for l in labels if l != agent_label]

        # baselines
        for lab in baseline_labels:
            line, = ax.plot(
                xs, series[lab],
                marker=mk.get(lab, "o"),
                linewidth=1.0,
                markersize=3.0,
                alpha=0.65,
                label=lab
            )
            if lab not in handles_for_legend:
                handles_for_legend[lab] = line

        # agent
        if agent_label is not None:
            line, = ax.plot(
                xs, series[agent_label],
                marker=mk.get("Agent1", "D"),
                linewidth=2.0,
                markersize=3.8,
                label="Agent1"
            )
            if "Agent1" not in handles_for_legend:
                handles_for_legend["Agent1"] = line

        ax.set_title(f"{letter} {panel_titles[key]}", fontsize=FS_TITLE, pad=2)

        ax.xaxis.set_minor_locator(AutoMinorLocator(2))
        ax.yaxis.set_minor_locator(AutoMinorLocator(2))
        ax.tick_params(axis="both", which="major", length=3.0, pad=1.2, labelsize=FS_TICK)
        ax.tick_params(axis="both", which="minor", length=1.6)

        ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.35)
        ax.grid(True, which="minor", linestyle=":", linewidth=0.4, alpha=0.25)

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(0, global_ymax * 1.05)

    # Hide rightmost x tick label on first two panels to avoid overlap at panel borders
    axes[0].xaxis.set_major_formatter(FuncFormatter(fmt_hide_rightmost))
    axes[1].xaxis.set_major_formatter(FuncFormatter(fmt_hide_rightmost))
    axes[2].xaxis.set_major_formatter(FormatStrFormatter("%.2f"))

    # labels: shared x-label
    axes[0].set_ylabel("MAE", fontsize=FS_LABEL, labelpad=2)
    fig.supxlabel("Noise level (nl)", fontsize=FS_LABEL, y=0.10)

    # legend order: Agent1 first, then others sorted
    legend_labels = list(handles_for_legend.keys())
    if "Agent1" in legend_labels:
        legend_labels.remove("Agent1")
        legend_labels = ["Agent1"] + sorted(legend_labels)
    else:
        legend_labels = sorted(legend_labels)

    legend_handles = [handles_for_legend[l] for l in legend_labels]

    # Put legend into dedicated legend row axis (never overlaps plots)
    leg = ax_leg.legend(
        legend_handles, legend_labels,
        loc="center",
        ncol=4,
        fontsize=FS_LEGEND,
        frameon=True,
        fancybox=False,
        framealpha=0.9,
        borderpad=0.25,
        handlelength=2.0,
        handletextpad=0.45,
        columnspacing=0.9,
        markerscale=1.2
    )
    leg.get_frame().set_linewidth(0.6)

    # Save: shrink padding to almost zero
    fig.savefig(OUT_PNG, dpi=600, bbox_inches="tight", pad_inches=0.003)
    fig.savefig(OUT_PDF, bbox_inches="tight", pad_inches=0.003)
    plt.close(fig)

    print("Saved:")
    print(" -", OUT_PNG)
    print(" -", OUT_PDF)


if __name__ == "__main__":
    main()


Saved:
 - exp01_all_noise_agent1_vs_baselines_mae_vs_nl_ijcai.png
 - exp01_all_noise_agent1_vs_baselines_mae_vs_nl_ijcai.pdf
